# Final FST scan

This notebook generates the final FST dataset used in later analysis.

It follows the same MalariaGEN setup used in the earlier project notebooks:

- `pandas==2.2.2`
- `malariagen_data>=15,<16`
- `scikit-allel`
- `ag3 = malariagen_data.Ag3()`

The analysis uses four 2022 female *Anopheles coluzzii* populations with at least 50 individuals, all six population pairs, chromosomes X and 3R, and 0-fold and 4-fold coding sites in non-overlapping 1 Mb windows.

The scan is checkpointed so it can be resumed after a Colab restart.

Also the version nedd to be 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

## Setup

In [2]:
# Check the Python runtime version to ensure compatibility with malariagen_data and other dependencies
import sys
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
from google.colab import drive
drive.mount("/content/drive")

# Quietly install specific versions of pandas, malariagen_data, and scikit-allel
%pip install -q "pandas==2.2.2" "malariagen_data>=15,<16" scikit-allel

import gc
import os
import itertools
import numpy as np
import pandas as pd
import malariagen_data

# All outputs from this notebook are written here.
OUTPUT_DIR = "/content/drive/MyDrive/FYP/final_output"

DATA_DIR = os.path.join(OUTPUT_DIR, "data")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")

for folder in [OUTPUT_DIR, DATA_DIR, TABLE_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Output folder:", OUTPUT_DIR)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 43.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## Connect to the MalariaGEN Ag3 resource

In [4]:
ag3 = malariagen_data.Ag3()

# Load sample metadata before selecting the four populations.
metadata = ag3.sample_metadata()

print("Metadata rows:", len(metadata))
print("Metadata columns:", len(metadata.columns))

Metadata rows: 25230
Metadata columns: 58


## Define the four final populations

In [5]:
# Define target population specifications (ID, short ID, country, and specific location)

population_specs = [
    {
        "Population_ID": "Nigeria_Gombe",
        "Short_ID": "NG_Gombe",
        "Country": "Nigeria",
        "Location": "Gombe",
    },
    {
        "Population_ID": "BurkinaFaso_BanaVillage",
        "Short_ID": "BF_BanaVil",
        "Country": "Burkina Faso",
        "Location": "Bana Village",
    },
    {
        "Population_ID": "Guinea_Siguiri_Dankakoro",
        "Short_ID": "GN_Siguiri",
        "Country": "Guinea",
        "Location": "Siguiri_Dankakoro",
    },
    {
        "Population_ID": "Mali_Faladje",
        "Short_ID": "ML_Faladje",
        "Country": "Mali",
        "Location": "Faladje",
    },
]

selected_populations = []

for spec in population_specs:
    # Use the same metadata filters for every population.
    query = (
        "taxon == 'coluzzii' and year == 2022 and sex_call == 'F' "
        f"and country == {repr(spec['Country'])} "
        f"and location == {repr(spec['Location'])}"
    )

    samples = metadata.query(query).copy()

    # The analysis excludes populations with fewer than 50 individuals.
    if len(samples) < 50:
        raise ValueError(
            f"{spec['Population_ID']} has only {len(samples)} individuals."
        )

    selected_populations.append({
        **spec,
        "N_individuals": len(samples),
        "Latitude": samples["latitude"].iloc[0],
        "Longitude": samples["longitude"].iloc[0],
        "Sample_query": query,
    })

populations = pd.DataFrame(selected_populations)

populations.to_csv(
    os.path.join(DATA_DIR, "selected_4populations_summary.csv"),
    index=False,
)

display(populations)

,Population_ID,Short_ID,Country,Location,N_individuals,Latitude,Longitude,Sample_query
0,Nigeria_Gombe,NG_Gombe,Nigeria,Gombe,57,10.298,11.173,taxon == 'coluzzii' and year == 2022 and sex_c...
1,BurkinaFaso_BanaVillage,BF_BanaVil,Burkina Faso,Bana Village,120,11.233,-4.472,taxon == 'coluzzii' and year == 2022 and sex_c...
2,Guinea_Siguiri_Dankakoro,GN_Siguiri,Guinea,Siguiri_Dankakoro,81,11.416,-9.058,taxon == 'coluzzii' and year == 2022 and sex_c...
3,Mali_Faladje,ML_Faladje,Mali,Faladje,75,13.137,-8.338,taxon == 'coluzzii' and year == 2022 and sex_c...


## Save the exact selected samples

In [6]:
# Keeping the exact selected sample metadata provides a provenance record and allows later checks of population size and coordinates.
selected_samples = []

for _, pop in populations.iterrows():
    samples = metadata.query(pop["Sample_query"]).copy()
    samples["Population_ID"] = pop["Population_ID"]
    selected_samples.append(samples)

selected_samples_exact = pd.concat(
    selected_samples,
    ignore_index=True,
)

selected_samples_exact.to_csv(
    os.path.join(DATA_DIR, "selected_samples_exact.csv"),
    index=False,
)

print("Total selected samples:", len(selected_samples_exact))
print(
    selected_samples_exact.groupby("Population_ID")
    .size()
    .sort_values(ascending=False)
)

Total selected samples: 333
Population_ID
BurkinaFaso_BanaVillage     120
Guinea_Siguiri_Dankakoro     81
Mali_Faladje                 75
Nigeria_Gombe                57
dtype: int64


## Create all six population pairs

In [7]:
# Fixed pair order keeps figures and tables consistent across the project.
pair_order = [
    ("Nigeria_Gombe", "BurkinaFaso_BanaVillage"),
    ("Nigeria_Gombe", "Guinea_Siguiri_Dankakoro"),
    ("Nigeria_Gombe", "Mali_Faladje"),
    ("BurkinaFaso_BanaVillage", "Guinea_Siguiri_Dankakoro"),
    ("BurkinaFaso_BanaVillage", "Mali_Faladje"),
    ("Guinea_Siguiri_Dankakoro", "Mali_Faladje"),
]

pair_rows = []
# Build metadata for each pair (labels, sample sizes, and group classification)
for pop1_id, pop2_id in pair_order:
    p1 = populations.query("Population_ID == @pop1_id").iloc[0]
    p2 = populations.query("Population_ID == @pop2_id").iloc[0]

    pair_rows.append({
        "Pair_ID": f"{pop1_id}_vs_{pop2_id}",
        "Pair_short": f"{p1['Short_ID']}_vs_{p2['Short_ID']}",
        "Pair_label": f"{p1['Short_ID']} vs {p2['Short_ID']}",
        "Population_1": pop1_id,
        "Population_2": pop2_id,
        "Country_1": p1["Country"],
        "Country_2": p2["Country"],
        "Location_1": p1["Location"],
        "Location_2": p2["Location"],
        "N_population_1": int(p1["N_individuals"]),
        "N_population_2": int(p2["N_individuals"]),
        "Comparison_group": (
            "With Nigeria_Gombe"
            if "Nigeria_Gombe" in [pop1_id, pop2_id]
            else "Without Nigeria"
        ),
        "Cohort1_query": p1["Sample_query"],
        "Cohort2_query": p2["Sample_query"],
    })

pairs = pd.DataFrame(pair_rows)

pairs.to_csv(
    os.path.join(DATA_DIR, "selected_6pairs.csv"),
    index=False,
)

display(pairs)

,Pair_ID,Pair_short,Pair_label,Population_1,Population_2,Country_1,Country_2,Location_1,Location_2,N_population_1,N_population_2,Comparison_group,Cohort1_query,Cohort2_query
0,Nigeria_Gombe_vs_BurkinaFaso_BanaVillage,NG_Gombe_vs_BF_BanaVil,NG_Gombe vs BF_BanaVil,Nigeria_Gombe,BurkinaFaso_BanaVillage,Nigeria,Burkina Faso,Gombe,Bana Village,57,120,With Nigeria_Gombe,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...
1,Nigeria_Gombe_vs_Guinea_Siguiri_Dankakoro,NG_Gombe_vs_GN_Siguiri,NG_Gombe vs GN_Siguiri,Nigeria_Gombe,Guinea_Siguiri_Dankakoro,Nigeria,Guinea,Gombe,Siguiri_Dankakoro,57,81,With Nigeria_Gombe,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...
2,Nigeria_Gombe_vs_Mali_Faladje,NG_Gombe_vs_ML_Faladje,NG_Gombe vs ML_Faladje,Nigeria_Gombe,Mali_Faladje,Nigeria,Mali,Gombe,Faladje,57,75,With Nigeria_Gombe,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...
3,BurkinaFaso_BanaVillage_vs_Guinea_Siguiri_Dank...,BF_BanaVil_vs_GN_Siguiri,BF_BanaVil vs GN_Siguiri,BurkinaFaso_BanaVillage,Guinea_Siguiri_Dankakoro,Burkina Faso,Guinea,Bana Village,Siguiri_Dankakoro,120,81,Without Nigeria,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...
4,BurkinaFaso_BanaVillage_vs_Mali_Faladje,BF_BanaVil_vs_ML_Faladje,BF_BanaVil vs ML_Faladje,BurkinaFaso_BanaVillage,Mali_Faladje,Burkina Faso,Mali,Bana Village,Faladje,120,75,Without Nigeria,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...
5,Guinea_Siguiri_Dankakoro_vs_Mali_Faladje,GN_Siguiri_vs_ML_Faladje,GN_Siguiri vs ML_Faladje,Guinea_Siguiri_Dankakoro,Mali_Faladje,Guinea,Mali,Siguiri_Dankakoro,Faladje,81,75,Without Nigeria,taxon == 'coluzzii' and year == 2022 and sex_c...,taxon == 'coluzzii' and year == 2022 and sex_c...


## Define the 1 Mb X and 3R windows

In [8]:
# Coordinates are counted from the centromeric end of each chromosome arm.
# 3R provides 25 windows over the first 25 Mb.
# X is shorter, so only windows with valid positive genomic coordinates are retained.

ARM_ENDS = {
    "3R": 53_200_684,
    "X": 24_393_108,
}

WINDOW_SIZE_BP = 1_000_000
MAX_DISTANCE_FROM_CEN_BP = 25_000_000

window_rows = []
# Slice non-overlapping 1 Mb windows starting from the centromeric end
for contig, arm_end in ARM_ENDS.items():
    for distance_start in range(
        0,
        MAX_DISTANCE_FROM_CEN_BP,
        WINDOW_SIZE_BP,
    ):
        distance_end = distance_start + WINDOW_SIZE_BP

        genomic_start = arm_end - distance_end + 1
        genomic_end = arm_end - distance_start

        # shorter X chromosome does not have a full 25th 1 Mb window within positive coordinates.
        if genomic_start < 1:
            continue

        window_rows.append({
            "Contig": contig,
            "Region": f"{contig}:{genomic_start}-{genomic_end}",
            "Genomic_start_POS": genomic_start,
            "Genomic_end_POS": genomic_end,
            "Distance_from_CEN_start": distance_start,
            "Distance_from_CEN_end": distance_end,
            "Distance_from_CEN_midpoint_Mb":
                (distance_start + distance_end) / 2 / 1e6,
            "Window_size_bp": WINDOW_SIZE_BP,
        })

windows = pd.DataFrame(window_rows)

windows.to_csv(
    os.path.join(DATA_DIR, "scan_windows_1Mb_from_CEN_3R_X.csv"),
    index=False,
)

print("Unique windows by chromosome:")
print(windows.groupby("Contig")["Region"].nunique())
display(windows.head())

Unique windows by chromosome:
Contig
3R    25
X     24
Name: Region, dtype: int64


,Contig,Region,Genomic_start_POS,Genomic_end_POS,Distance_from_CEN_start,Distance_from_CEN_end,Distance_from_CEN_midpoint_Mb,Window_size_bp
0,3R,3R:52200685-53200684,52200685,53200684,0,1000000,0.5,1000000
1,3R,3R:51200685-52200684,51200685,52200684,1000000,2000000,1.5,1000000
2,3R,3R:50200685-51200684,50200685,51200684,2000000,3000000,2.5,1000000
3,3R,3R:49200685-50200684,49200685,50200684,3000000,4000000,3.5,1000000
4,3R,3R:48200685-49200684,48200685,49200684,4000000,5000000,4.5,1000000


## Check the expected number of FST jobs

In [9]:
# Every job is one population pair * one genomic window * one site class.
SITE_CLASSES = {
    "0-fold": "CDS_DEG_0",
    "4-fold": "CDS_DEG_4",
}

expected_jobs = len(pairs) * len(windows) * len(SITE_CLASSES)

print("Population pairs:", len(pairs))
print("Windows:", len(windows))
print("Site classes:", len(SITE_CLASSES))
print("Expected FST jobs:", expected_jobs)

Population pairs: 6
Windows: 49
Site classes: 2
Expected FST jobs: 588


## Run the checkpointed FST scan

In [ ]:
# Results are saved after every job so the notebook can continue from the last completed calculation
CHECKPOINT_FILE = os.path.join(
    DATA_DIR,
    "FST_selected6pairs_1Mb_3R_X_full_scan_minN50_checkpoint.csv",
)

FINAL_FILE = os.path.join(
    DATA_DIR,
    "FST_selected6pairs_1Mb_3R_X_full_scan_minN50.csv",
)

FST_JOBS_PER_RUN = 10

# Keep False during normal use.
FORCE_RECOMPUTE_FST = False

if FORCE_RECOMPUTE_FST:
    fst_results = pd.DataFrame()

elif os.path.exists(CHECKPOINT_FILE):
    fst_results = pd.read_csv(CHECKPOINT_FILE)
    print("Checkpoint loaded:", len(fst_results), "rows")

elif os.path.exists(FINAL_FILE):
    fst_results = pd.read_csv(FINAL_FILE)
    print("Final file loaded:", len(fst_results), "rows")

else:
    fst_results = pd.DataFrame()
    print("No previous result file found. Starting from scratch.")

if len(fst_results):
    completed_jobs = set(
        zip(
            fst_results["Pair_ID"].astype(str),
            fst_results["Region"].astype(str),
            fst_results["Site_class"].astype(str),
        )
    )
else:
    completed_jobs = set()

# Iterate through combinations of population pairs, genomic windows, and site classes
jobs_run_now = 0
stop_now = False

for _, pair in pairs.iterrows():
    if stop_now:
        break

    for _, window in windows.iterrows():
        if stop_now:
            break

        for site_class, site_class_argument in SITE_CLASSES.items():
            job_key = (
                str(pair["Pair_ID"]),
                str(window["Region"]),
                site_class,
            )

            # Skip work that is already present in the checkpoint.
            if job_key in completed_jobs:
                continue

            # Stop after the requested number of new jobs.
            if jobs_run_now >= FST_JOBS_PER_RUN:
                stop_now = True
                break

            print(
                f"{jobs_run_now + 1}/{FST_JOBS_PER_RUN} | "
                f"{pair['Pair_label']} | "
                f"{window['Region']} | "
                f"{site_class}"
            )

            try:
                # average_fst returns the window FST estimate and a block-jackknife standard error.
                fst_value, fst_se = ag3.average_fst(
                    region=window["Region"],
                    cohort1_query=pair["Cohort1_query"],
                    cohort2_query=pair["Cohort2_query"],
                    site_mask="default",
                    site_class=site_class_argument,
                    min_cohort_size=50,
                    max_cohort_size=None,
                    n_jack=20,
                    random_seed=42,
                )

                run_status = "success"
                error_message = ""

            except Exception as exc:
                # Failed jobs are kept in the table so the failure is visible.
                fst_value = np.nan
                fst_se = np.nan
                run_status = "failed"
                error_message = str(exc)

            result_row = {
                "Comparison_group": pair["Comparison_group"],
                "Pair_ID": pair["Pair_ID"],
                "Pair_short": pair["Pair_short"],
                "Pair_label": pair["Pair_label"],
                "Population_1": pair["Population_1"],
                "Population_2": pair["Population_2"],
                "Country_1": pair["Country_1"],
                "Country_2": pair["Country_2"],
                "Location_1": pair["Location_1"],
                "Location_2": pair["Location_2"],
                "N_population_1": pair["N_population_1"],
                "N_population_2": pair["N_population_2"],
                "Contig": window["Contig"],
                "Region": window["Region"],
                "Genomic_start_POS": window["Genomic_start_POS"],
                "Genomic_end_POS": window["Genomic_end_POS"],
                "Distance_from_CEN_start":
                    window["Distance_from_CEN_start"],
                "Distance_from_CEN_end":
                    window["Distance_from_CEN_end"],
                "Distance_from_CEN_midpoint_Mb":
                    window["Distance_from_CEN_midpoint_Mb"],
                "Window_size_bp": window["Window_size_bp"],
                "Site_class": site_class,
                "FST_estimate": fst_value,
                "Jackknife_standard_error": fst_se,
                "Run_status": run_status,
                "Error_message": error_message,
            }

            fst_results = pd.concat(
                [fst_results, pd.DataFrame([result_row])],
                ignore_index=True,
            )

            # Save after every calculation.
            fst_results.to_csv(
                CHECKPOINT_FILE,
                index=False,
            )

            completed_jobs.add(job_key)
            jobs_run_now += 1

            # Release large temporary objects between jobs.
            gc.collect()

print()
print("Completed unique jobs:", len(completed_jobs), "/", expected_jobs)

if len(completed_jobs) >= expected_jobs:
    fst_results.to_csv(
        FINAL_FILE,
        index=False,
    )

    print("Full FST scan is complete.")
    print("Final file:", FINAL_FILE)

else:
    print("This batch is complete.")
    print("Rerun the notebook to continue from the checkpoint.")

Checkpoint loaded: 62 rows
1/10 | NG_Gombe vs BF_BanaVil | X:17393109-18393108 | 0-fold
Access SNP calls: ⠏ (0:00:25.48)

Locate CDS_DEG_0 sites:   0%|          | 0/10 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/167 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/79 [00:00<?, ?it/s]

2/10 | NG_Gombe vs BF_BanaVil | X:17393109-18393108 | 4-fold
Access SNP calls: ⠋ (0:00:10.47)

...

Locate CDS_DEG_0 sites:   0%|          | 0/7 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/140 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/67 [00:00<?, ?it/s]

4/10 | NG_Gombe vs BF_BanaVil | X:16393109-17393108 | 4-fold
Access SNP calls: ⠦ (0:00:09.92)

Locate CDS_DEG_4 sites:   0%|          | 0/7 [00:00<?, ?it/s]

## Validate the completed scan

In [ ]:
# This cell run after every batch.
# When the scan is complete, it shows six pairs 24 X windows, 25 3R windows and two site classes.

result_file = (
    FINAL_FILE
    if os.path.exists(FINAL_FILE)
    else CHECKPOINT_FILE
)

scan = pd.read_csv(result_file)

successful = scan[
    scan["Run_status"].eq("success")
].copy()

validation = pd.DataFrame([{
    "Rows_in_file": len(scan),
    "Successful_rows": len(successful),
    "Failed_rows": int((scan["Run_status"] != "success").sum()),
    "Unique_pairs": successful["Pair_ID"].nunique(),
    "Unique_X_windows": successful.loc[
        successful["Contig"].eq("X"),
        "Region"
    ].nunique(),
    "Unique_3R_windows": successful.loc[
        successful["Contig"].eq("3R"),
        "Region"
    ].nunique(),
    "Site_classes": successful["Site_class"].nunique(),
}])

validation.to_csv(
    os.path.join(TABLE_DIR, "final_FST_scan_validation.csv"),
    index=False,
)

display(validation)

failed = scan[
    ~scan["Run_status"].eq("success")
][[
    "Pair_label",
    "Region",
    "Site_class",
    "Error_message",
]]

if len(failed):
    print("Failed jobs:")
    display(failed)
else:
    print("No failed jobs recorded.")